In [1]:
import pandas as pd

import numpy as np
from scipy.stats import spearmanr, pearsonr, kendalltau
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, median_absolute_error, explained_variance_score, max_error

from rogi import RoughnessIndex

from multifidelity_end2end import create_low_fidelity
import argparse

from rdkit import Chem
from rdkit.Chem import Descriptors

import matplotlib.pyplot as plt
import seaborn as sns

/home/kpg/miniconda3/envs/chemprop/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Define Metrics

In [2]:
def calculate_percentage_metrics(list1, list2):
    arr1 = np.array(list1)
    arr2 = np.array(list2)
    
    # Avoid division by zero; handle cases where actual values are zero
    arr1_safe = np.where(arr1 == 0, np.finfo(float).eps, arr1)  # Replace 0 with a very small number
    
    # Calculate MAPE - Mean Absolute Percentage Error
    mape = np.mean(np.abs((arr1 - arr2) / arr1_safe)) * 100
    
    # Calculate wMAPE - Weighted Mean Absolute Percentage Error
    wmape = np.sum(np.abs(arr1 - arr2)) / np.sum(arr1_safe) * 100
    
    # Include previous metrics for completeness
    mae = mean_absolute_error(arr1, arr2)
    mse = mean_squared_error(arr1, arr2)
    rmse = np.sqrt(mse)
    r_squared = r2_score(arr1, arr2)
    median_ae = median_absolute_error(arr1, arr2)
    explained_variance = explained_variance_score(arr1, arr2)
    max_err = max_error(arr1, arr2)
    spearman_corr, _ = spearmanr(arr1, arr2)
    pearson_corr, _ = pearsonr(arr1, arr2)
    kendall_corr, _ = kendalltau(arr1, arr2)
    covariance_matrix = np.cov(arr1, arr2)
    covariance = covariance_matrix[0, 1]
    corrcoef_matrix = np.corrcoef(arr1, arr2)
    corrcoef = corrcoef_matrix[0, 1]

    
    # Print results
    print(f"MAPE (Mean Absolute Percentage Error): {mape}%")
    print(f"wMAPE (Weighted Mean Absolute Percentage Error): {wmape}%")
    print(f"MAE (Mean Absolute Error): {mae}")
    print(f"RMSE (Root Mean Squared Error): {rmse}")
    print(f"R-squared: {r_squared}")
    print(f"Median Absolute Error: {median_ae}")
    print(f"Explained Variance Score: {explained_variance}")
    print(f"Max Error: {max_err}")
    print(f"Spearman's rank correlation: {spearman_corr}")
    print(f"Pearson correlation coefficient: {pearson_corr}")
    print(f"Kendall Tau correlation: {kendall_corr}")
    print(f"Covariance: {covariance}")
    print(f"Correlation coefficient: {corrcoef}")

    return mape, wmape, mae, rmse, r_squared, median_ae, explained_variance, max_err, spearman_corr, pearson_corr, kendall_corr, covariance, corrcoef

In [3]:
def compute_rogis(hf, lf, smiles, X=None, metric='tanimoto', verbose=False):
    ri_hf = RoughnessIndex(Y=hf, X=X, smiles=smiles, metric=metric, verbose=verbose)
    rogi_hf = ri_hf.compute_index()
    print(f"ROGI for HF: {rogi_hf}")
    ri_lf = RoughnessIndex(Y=lf, X=X, smiles=smiles, metric=metric, verbose=verbose)
    rogi_lf = ri_lf.compute_index()
    print(f"ROGI for LF: {rogi_lf}")
    ri_diff = RoughnessIndex(Y=hf-lf, X=X, smiles=smiles, metric=metric, verbose=verbose)
    rogi_diff = ri_diff.compute_index()
    print(f"ROGI for HF-LF: {rogi_diff}")
    ri_absdiff = RoughnessIndex(Y=abs(hf-lf), X=X, smiles=smiles, metric=metric, verbose=verbose)
    rogi_absdiff = ri_absdiff.compute_index()
    print(f"ROGI for |HF-LF|: {rogi_absdiff}")
    
    return rogi_hf, rogi_lf, rogi_diff, rogi_absdiff

In [4]:
# create a dataframe to store the results
column_list = ['dataset', 'mape', 'wmape', 'mae', 'rmse', 'r_squared', 'median_ae', 
                'explained_variance', 'max_err', 'spearman_corr', 'pearson_corr', 'kendall_corr', 'covariance', 
                'corrcoef', 'rogi_hf', 'rogi_lf', 'rogi_diff', 'rogi_absdiff', 
                # 'rogi_hf_desc_euc', 'rogi_lf_desc_euc', 
                # 'rogi_diff_desc_euc', 'rogi_absdiff_desc_euc', 'rogi_hf_desc_cos', 'rogi_lf_desc_cos', 'rogi_diff_desc_cos', 'rogi_absdiff_desc_cos',
                ]
results_df = pd.DataFrame(columns=column_list)

datasets = [
    ('lambda', 'tests/data/lambda_dft_expt.csv', 'peakwavs_max', 'energy_max_osc_nm'),
    ('drug', 'tests/data/pubchem_AID1445_sd_dr_0.1.csv', 'DR', 'SD'),
    ('sol', 'tests/data/SolProp_v1.2_Data/dHsolv_cosmo_expt.csv', 'dHsolv_expt', 'dHsolv_cosmo'),
            ]

for dataset in datasets:   
    dataset_name = dataset[0]         
    df = pd.read_csv(dataset[1])

    print(f'Dataset name: {dataset_name}')
    print("Percentage Metrics")
    mape, wmape, mae, rmse, r_squared, median_ae, explained_variance, \
        max_err, spearman_corr, pearson_corr, kendall_corr, covariance, corrcoef = calculate_percentage_metrics(df[dataset[2]], df[dataset[3]])
    print()
    print("ROGIs")
    rogi_hf, rogi_lf, rogi_diff, rogi_absdiff = compute_rogis(df[dataset[2]], df[dataset[3]], df['smiles'])
    # print()
    # print('Euclidean distance - descriptors')
    # rogi_hf_desc_euc, rogi_lf_desc_euc, rogi_diff_desc_euc, rogi_absdiff_desc_euc = compute_rogis(df[dataset[2]], df[dataset[3]], df['smiles'], X=desc_df, metric='euclidean')
    # print()
    # print('Cosine distance - descriptors')
    # rogi_hf_desc_cos, rogi_lf_desc_cos, rogi_diff_desc_cos, rogi_absdiff_desc_cos = compute_rogis(df[dataset[2]], df[dataset[3]], df['smiles'], X=desc_df, metric='cosine')
    print()
    print()

    # store all the results in a dataframe
    results_df = pd.concat([results_df, pd.DataFrame([[dataset_name,
                                                mape, wmape, mae, rmse, r_squared, median_ae, explained_variance, 
                                                max_err, spearman_corr, pearson_corr, kendall_corr, covariance, corrcoef,
                                                rogi_hf, rogi_lf, rogi_diff, rogi_absdiff, 
                                                # rogi_hf_desc_euc, rogi_lf_desc_euc, 
                                                # rogi_diff_desc_euc, rogi_absdiff_desc_euc, rogi_hf_desc_cos, rogi_lf_desc_cos, 
                                                # rogi_diff_desc_cos, rogi_absdiff_desc_cos,
                                                ]], columns=column_list)])   

results_df

Dataset name: lambda
Percentage Metrics
MAPE (Mean Absolute Percentage Error): 15.83795005552309%
wMAPE (Weighted Mean Absolute Percentage Error): 16.809480525459318%
MAE (Mean Absolute Error): 75.15328665192277
RMSE (Root Mean Squared Error): 94.21452198109924
R-squared: 0.03668773205848175
Median Absolute Error: 63.37383602089483
Explained Variance Score: 0.5834573211690091
Max Error: 995.1428571428571
Spearman's rank correlation: 0.8183146547209219
Pearson correlation coefficient: 0.7666716574313159
Kendall Tau correlation: 0.6522124289874525
Covariance: 4952.348938570619
Correlation coefficient: 0.7666716574313157

ROGIs
ROGI for HF: 0.021435714797693572
ROGI for LF: 0.01232026104303921
ROGI for HF-LF: 0.01063191321568116
ROGI for |HF-LF|: 0.016085917612476935


Dataset name: drug
Percentage Metrics


KeyError: 'dHsolv_expt'

In [ ]:
raise

# Plots

In [ ]:
results_df = pd.read_csv('h298_dataset_analysis_results.csv')

In [ ]:
results_df.iloc[[0,1,7]]

In [ ]:
# make a grouped bar plot of the ROGI values for different values of polynomial bias
df_pivot = pd.pivot_table(
	results_df.iloc[[0,1,2,3]],
	values=['rogi_hf', 'rogi_lf', 'rogi_diff', 'rogi_absdiff'],
	index=['add_pn_bias_to_make_lf'],
	aggfunc=np.mean
)

df_pivot = df_pivot.reset_index()
df_pivot = df_pivot.melt(id_vars='add_pn_bias_to_make_lf', value_vars=['rogi_hf', 'rogi_lf', 'rogi_diff', 'rogi_absdiff'], var_name='ROGI', value_name='Value')

plt.figure(figsize=(10, 6))
sns.barplot(x='add_pn_bias_to_make_lf', y='Value', hue='ROGI', data=df_pivot)
plt.xlabel('Polynomial Bias')
plt.ylabel('ROGI')
plt.show()

# make a grouped bar plot of the ROGI values for different values of constant bias
df_pivot = pd.pivot_table(
	results_df.iloc[[0,4,5]],
	values=['rogi_hf', 'rogi_lf', 'rogi_diff', 'rogi_absdiff'],
	index=['add_constant_bias_to_make_lf'],
	aggfunc=np.mean
)

df_pivot = df_pivot.reset_index()
df_pivot = df_pivot.melt(id_vars='add_constant_bias_to_make_lf', value_vars=['rogi_hf', 'rogi_lf', 'rogi_diff', 'rogi_absdiff'], var_name='ROGI', value_name='Value')

plt.figure(figsize=(10, 6))
sns.barplot(x='add_constant_bias_to_make_lf', y='Value', hue='ROGI', data=df_pivot)
plt.xlabel('Constant Bias')
plt.ylabel('ROGI')
plt.show()

# make a grouped bar plot of the ROGI values for different values of Gaussian noise
df_pivot = pd.pivot_table(
	results_df.iloc[[0,6,7]],
	values=['rogi_hf', 'rogi_lf', 'rogi_diff', 'rogi_absdiff'],
	index=['add_gauss_noise_to_make_lf'],
	aggfunc=np.mean
)

df_pivot = df_pivot.reset_index()
df_pivot = df_pivot.melt(id_vars='add_gauss_noise_to_make_lf', value_vars=['rogi_hf', 'rogi_lf', 'rogi_diff', 'rogi_absdiff'], var_name='ROGI', value_name='Value')

plt.figure(figsize=(10, 6))
sns.barplot(x='add_gauss_noise_to_make_lf', y='Value', hue='ROGI', data=df_pivot)
plt.xlabel('Gaussian Noise')
plt.ylabel('ROGI')
plt.show()

# make a grouped bar plot of the ROGI values for different values of descriptor bias
df_pivot = pd.pivot_table(
	results_df.iloc[[0,8,9]],
	values=['rogi_hf', 'rogi_lf', 'rogi_diff', 'rogi_absdiff'],
	index=['add_descriptor_bias_to_make_lf'],
	aggfunc=np.mean
)

df_pivot = df_pivot.reset_index()
df_pivot = df_pivot.melt(id_vars='add_descriptor_bias_to_make_lf', value_vars=['rogi_hf', 'rogi_lf', 'rogi_diff', 'rogi_absdiff'], var_name='ROGI', value_name='Value')

plt.figure(figsize=(10, 6))
sns.barplot(x='add_descriptor_bias_to_make_lf', y='Value', hue='ROGI', data=df_pivot)
plt.xlabel('Descriptor Bias')
plt.ylabel('ROGI')
plt.show()